# 02 — TON_IoT Cleaning and Preprocessing

## Purpose

This notebook turns the audited network-flow data into model-ready training, validation, and test sets. It provides the evidence for the report section **Data Preparation and Feature Engineering**.


In [1]:
from pathlib import Path
import json
import warnings
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# Works whether you run from the repo root or from notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
DATA_PATH = DATA_DIR / "train_test_network.csv"
if not DATA_PATH.exists():
    alternatives = list(DATA_DIR.glob("*.csv"))
    if alternatives:
        DATA_PATH = alternatives[0]
    else:
        raise FileNotFoundError(f"No CSV found in {DATA_DIR}. Update DATA_PATH with the exact filename.")

FIGURES_DIR = PROJECT_ROOT / "results" / "figures"
METRICS_DIR = PROJECT_ROOT / "results" / "metrics"
MODELS_DIR = PROJECT_ROOT / "models"
for folder in (FIGURES_DIR, METRICS_DIR, MODELS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print("Dataset:", DATA_PATH)

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


Dataset: data\train_test_network.csv


In [2]:
df = pd.read_csv(DATA_PATH)
print("Loaded:", df.shape)

identifier_columns = [c for c in ["src_ip", "dst_ip", "uid", "id", "flow_id", "timestamp"] if c in df.columns]
drop_columns = ["label", "type"] + identifier_columns
X = df.drop(columns=drop_columns, errors="ignore")
y_binary = df["label"].astype(int)
y_multiclass = df["type"].astype(str)

print("Removed from main feature matrix:", identifier_columns)
print("Remaining features:", X.shape[1])


Loaded: (211043, 44)
Removed from main feature matrix: ['src_ip', 'dst_ip']
Remaining features: 40


## Why these fields are removed

The target columns cannot be used as predictors. IP addresses and identifiers are removed from the main model because they may identify a device, capture, or environment rather than represent general attack behavior.

## 1. Create stratified train, validation, and test sets

The baseline split is 70% training, 15% validation, and 15% testing. Stratification preserves the category proportions. This is the initial random split; a capture/device/time-disjoint split is a planned leakage study.


In [3]:
RANDOM_STATE = 42
X_train, X_temp, yb_train, yb_temp, ym_train, ym_temp = train_test_split(
    X, y_binary, y_multiclass, test_size=0.30, random_state=RANDOM_STATE, stratify=y_multiclass
)
X_val, X_test, yb_val, yb_test, ym_val, ym_test = train_test_split(
    X_temp, yb_temp, ym_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=ym_temp
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)
print("MITM counts:", int((ym_train == "mitm").sum()), int((ym_val == "mitm").sum()), int((ym_test == "mitm").sum()))


Train: (147730, 40)
Validation: (31656, 40)
Test: (31657, 40)
MITM counts: 730 156 157


## 2. Encode categorical features and prepare numerical features

The transformer is fitted only on training data. Numerical fields use median imputation and scaling. Categorical fields use the most frequent value and one-hot encoding. Unknown categories in validation or test data are ignored safely.


In [4]:
categorical_columns = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numerical_columns = X_train.select_dtypes(include=np.number).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numerical_columns),
    ("categorical", categorical_pipeline, categorical_columns),
], remainder="drop")

X_train_encoded = preprocessor.fit_transform(X_train)
X_val_encoded = preprocessor.transform(X_val)
X_test_encoded = preprocessor.transform(X_test)

print("Encoded train:", X_train_encoded.shape)
print("Encoded validation:", X_val_encoded.shape)
print("Encoded test:", X_test_encoded.shape)


Encoded train: (147730, 889)
Encoded validation: (31656, 889)
Encoded test: (31657, 889)


## 3. Save preprocessing objects and metadata

In [5]:
joblib.dump(preprocessor, MODELS_DIR / "preprocessor_main.joblib")

metadata = {
    "dataset": str(DATA_PATH),
    "removed_identifier_columns": identifier_columns,
    "categorical_columns": categorical_columns,
    "numerical_columns": numerical_columns,
    "train_rows": len(X_train),
    "validation_rows": len(X_val),
    "test_rows": len(X_test),
    "split": "70/15/15 stratified random split",
    "leakage_note": "IP/identifier fields excluded; grouped split remains future work",
}
with open(METRICS_DIR / "preprocessing_metadata.json", "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)
print("Saved preprocessing workflow and metadata.")


Saved preprocessing workflow and metadata.


## Report conclusion

The preprocessing workflow removes likely identifier leakage, separates the two targets, handles missing values, encodes categorical variables, scales numerical variables, and fits transformations using training data only. Class imbalance is preserved for evaluation; later model training can use class weighting on training data only.
